# CDL6000 - Επίδειξη Pipeline Επεξεργασίας Νομικών Εγγράφων

Αυτό το notebook παρουσιάζει τις βασικές λειτουργίες του συστήματος CDL6000 για την ανάλυση και ταξινόμηση νομικών εγγράφων.

## Λειτουργίες που θα επιδείξουμε:
1. Προεπεξεργασία νομικών εγγράφων
2. Εξαγωγή νομικών χαρακτηριστικών και παραπομπών
3. Ταξινόμηση κειμένων σε κατηγορίες
4. Μέτρηση απόδοσης και παρακολούθηση μνήμης

In [ ]:
# Εισαγωγή απαραίτητων βιβλιοθηκών
import sys
import os
from pathlib import Path
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time

# Ρύθμιση του project path
project_root = Path.cwd().parent  # Υποθέτουμε ότι το notebook είναι στον φάκελο notebooks/
sys.path.append(str(project_root))

# Ρύθμιση logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('CDL6000-Notebook')

In [ ]:
# Έλεγχος διαθεσιμότητας GPU
from src.core.test_cuda import test_cuda

try:
    test_cuda()
except Exception as e:
    print(f"Σφάλμα κατά τον έλεγχο GPU: {e}")
    print("Το σύστημα θα λειτουργήσει μόνο με CPU.")

## 1. Προεπεξεργασία Νομικών Εγγράφων

Θα χρησιμοποιήσουμε την κλάση `LegalDocumentPreprocessor` για να επεξεργαστούμε νομικά έγγραφα και να εξάγουμε βασικά χαρακτηριστικά.

In [ ]:
from src.core.preprocessor import LegalDocumentPreprocessor
from src.core.metrics import PerformanceTracker

# Δημιουργία του preprocessor
preprocessor = LegalDocumentPreprocessor(batch_size=16)

# Δημιουργία του performance tracker
performance_tracker = PerformanceTracker()

In [ ]:
# Δείγματα νομικών κειμένων
sample_legal_texts = [
    """
    [2015] EWHC 123 (Ch)
    In the matter of Smith v. Jones Corporation Ltd.
    This case concerns a breach of Article 5 of the Commercial Agreement Act of 2010.
    The Court finds that section 15, paragraph 3 of the contract was violated by the defendant.
    Prior cases include Brown v. Wilson [2012] UKSC 45 and Roberts v. Thompson [2008] EWCA Civ 23.
    """,
    """
    [2017] UKSC 72
    R (on the application of Miller) v Secretary of State for Exiting the European Union
    This landmark case considered whether the UK Government required Parliamentary approval before triggering Article 50.
    The Supreme Court held, by a majority of 8 to 3, that an Act of Parliament was required.
    Reference was made to R v Secretary of State for Transport, ex p Factortame Ltd [1990] 2 AC 85.
    """,
    """
    [2013] UKSC 34
    Prest v Petrodel Resources Ltd & Others
    The case concerns the circumstances in which the court may pierce the corporate veil.
    The judgment clarified the doctrine of piercing the corporate veil under Section 24 of the Matrimonial Causes Act 1973.
    Earlier cases of Salomon v Salomon & Co Ltd [1896] UKHL 1 and Adams v Cape Industries plc [1990] Ch 433 were considered.
    """
]

# Έναρξη του performance tracker
performance_tracker.start()

# Επεξεργασία κάθε κειμένου
results = []
for i, text in enumerate(sample_legal_texts):
    # Επεξεργασία του κειμένου
    features = preprocessor.process_document(text)
    results.append(features)
    
    # Καταγραφή της απόδοσης
    metrics = performance_tracker.record_document_processed(len(text))
    print(f"Κείμενο {i+1} - Χρόνος: {metrics['processing_time']:.3f}s, Μνήμη: {metrics['memory_usage_mb']:.2f} MB")

# Εμφάνιση των αποτελεσμάτων
for i, result in enumerate(results):
    print(f"\nΑποτελέσματα για το κείμενο {i+1}:")
    for key, value in result.items():
        print(f"  {key}: {value}")

In [ ]:
# Εμφάνιση συγκεντρωτικών μετρικών απόδοσης
summary = performance_tracker.get_summary()
print("\nΣυγκεντρωτικές μετρικές απόδοσης:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 2. Εξαγωγή Νομικών Όρων και Οντοτήτων

Θα χρησιμοποιήσουμε την κλάση `LegalTermExtractor` για να εξάγουμε νομικούς όρους και οντότητες από τα έγγραφα.

In [ ]:
from src.ml.legal_extractor import LegalTermExtractor

# Δημιουργία του extractor
extractor = LegalTermExtractor()

# Επεξεργασία κάθε κειμένου
legal_terms_results = []
for i, text in enumerate(sample_legal_texts):
    # Εξαγωγή νομικών όρων
    terms = extractor.extract_legal_terms(text)
    legal_terms_results.append(terms)
    
    print(f"\nΝομικοί όροι για το κείμενο {i+1}:")
    for term_type, matches in terms.items():
        print(f"  {term_type}: {matches}")

In [ ]:
# Πλήρης ανάλυση των κειμένων
analysis_results = []
for i, text in enumerate(sample_legal_texts):
    # Ανάλυση εγγράφου
    analysis = extractor.analyze_document(text)
    analysis_results.append(analysis)
    
    print(f"\nΠλήρης ανάλυση για το κείμενο {i+1}:")
    print(f"  Μέγεθος κειμένου: {analysis.get('text_length')}")
    print(f"  Αριθμός παραπομπών: {analysis.get('citation_count')}")
    print(f"  Έτη που αναφέρονται: {analysis.get('years_mentioned')}")
    print(f"  Εύρος ετών: {analysis.get('year_range')}")
    print(f"  Συνολικοί νομικοί όροι: {analysis.get('total_legal_terms')}")

## 3. Ταξινόμηση Νομικών Εγγράφων

Θα χρησιμοποιήσουμε την κλάση `LegalDocumentClassifier` για να εκπαιδεύσουμε ένα μοντέλο ταξινόμησης των νομικών εγγράφων σε κατηγορίες.

In [ ]:
from src.core.classifier import LegalDocumentClassifier
from src.core.metrics import ClassificationMetrics

# Δημιουργία του classifier
classifier = LegalDocumentClassifier(algorithm='svm', use_gpu=torch.cuda.is_available())

# Δημιουργία του metrics calculator
metrics_calc = ClassificationMetrics()

In [ ]:
# Δείγματα νομικών κειμένων για εκπαίδευση
train_texts = [
    "The court finds that the defendant violated section 15 of the contract.",
    "Pursuant to Article 10 of Regulation EC 2015/123, the claim is dismissed.",
    "The plaintiff filed a motion for summary judgment under Rule 56.",
    "The Supreme Court overturned the decision of the Court of Appeal.",
    "The parties have reached a settlement agreement on all claims.",
    "The Court of Appeal upheld the judgment of the lower court.",
    "The claimant seeks damages for breach of contract under common law.",
    "The case was referred to the European Court of Justice.",
    "The tribunal found that the company had violated labor regulations.",
    "The arbitration award was set aside by the High Court.",
    "The contract stipulates that any disputes will be resolved through arbitration.",
    "According to EU Directive 2019/1152, employees have the right to predictable working conditions.",
    "The judge granted an injunction preventing disclosure of confidential information.",
    "The House of Lords decision in Donoghue v Stevenson established the modern concept of negligence.",
    "The tenant claims breach of the covenant of quiet enjoyment under the lease agreement."
]

# Ετικέτες κατηγοριών για τα κείμενα εκπαίδευσης
train_labels = [
    "Contract", "Regulatory", "Procedural", "Appeal", "Settlement",
    "Appeal", "Contract", "Referral", "Employment", "Arbitration",
    "Contract", "Regulatory", "Injunction", "Tort", "Property"
]

# Δείγματα για δοκιμή
test_texts = [
    "The appellant contests the decision on grounds of procedural error.",
    "The agreement contains a force majeure clause that excuses non-performance.",
    "The worker alleges unfair dismissal contrary to employment regulations.",
    "The parties submitted their dispute to binding arbitration as per the contract."
]

# Πραγματικές ετικέτες για τα δείγματα δοκιμής
test_labels = ["Appeal", "Contract", "Employment", "Arbitration"]

In [ ]:
# Εκπαίδευση του ταξινομητή
print("Εκπαίδευση του μοντέλου SVM...")
train_metrics = classifier.train(train_texts, train_labels, max_features=1000)

print(f"\nΑποτελέσματα εκπαίδευσης:")
print(f"  Ακρίβεια: {train_metrics['accuracy']:.4f}")
print(f"  F1 Score: {train_metrics['f1_score']:.4f}")
print(f"  Χρόνος εκπαίδευσης: {train_metrics['training_time']:.2f} δευτερόλεπτα")

In [ ]:
# Δοκιμή του ταξινομητή
predictions = classifier.predict(test_texts)

print("\nΑποτελέσματα ταξινόμησης:")
for i, (text, pred, true) in enumerate(zip(test_texts, predictions, test_labels)):
    match = "✓" if pred == true else "✗"
    print(f"{i+1}. {text}\n   Πρόβλεψη: {pred}, Πραγματική: {true} {match}\n")

In [ ]:
# Υπολογισμός μετρικών ταξινόμησης
class_names = sorted(set(train_labels))
metrics_results = metrics_calc.calculate(test_labels, predictions, class_names)

print("\nΜετρικές ταξινόμησης:")
print(f"  Ακρίβεια: {metrics_results['accuracy']:.4f}")
print(f"  Precision: {metrics_results['precision']:.4f}")
print(f"  Recall: {metrics_results['recall']:.4f}")
print(f"  F1 Score: {metrics_results['f1_score']:.4f}")

# Οπτικοποίηση του πίνακα σύγχυσης
metrics_calc.plot_confusion_matrix(metrics_results, class_names, "SVM Classifier")

## 4. Χρήση του Specialized Legal Vectorizer

Θα χρησιμοποιήσουμε τον εξειδικευμένο vectorizer για νομικά κείμενα.

In [ ]:
from src.ml.vectorizer import LegalVectorizer

# Δημιουργία του legal vectorizer
legal_vectorizer = LegalVectorizer(max_features=1000, include_legal_features=True)

In [ ]:
# Μετατροπή των κειμένων σε διανύσματα
X_train = legal_vectorizer.fit_transform(train_texts)

print(f"Διαστάσεις των διανυσμάτων εκπαίδευσης: {X_train.shape}")

# Μετατροπή των κειμένων δοκιμής
X_test = legal_vectorizer.transform(test_texts)
print(f"Διαστάσεις των διανυσμάτων δοκιμής: {X_test.shape}")

In [ ]:
# Προβολή των κορυφαίων χαρακτηριστικών
feature_names = legal_vectorizer.get_feature_names()
print(f"Συνολικός αριθμός χαρακτηριστικών: {len(feature_names)}")

print("\nΚορυφαία χαρακτηριστικά:")
for i in range(min(20, len(feature_names))):
    print(f"  {i+1}. {feature_names[i]}")

print("\nΝομικά χαρακτηριστικά:")
legal_features = feature_names[-6:] if legal_vectorizer.include_legal_features else []
for i, feature in enumerate(legal_features):
    print(f"  {i+1}. {feature}")

## 5. Σύγκριση Διαφορετικών Αλγορίθμων Ταξινόμησης

Θα συγκρίνουμε διαφορετικούς αλγόριθμους ταξινόμησης για να βρούμε τον καλύτερο για το συγκεκριμένο πρόβλημα.

In [ ]:
# Λίστα αλγορίθμων για σύγκριση
algorithms = ['svm', 'naive_bayes', 'random_forest']
model_results = []

for algorithm in algorithms:
    print(f"\nΕκπαίδευση και αξιολόγηση του αλγορίθμου: {algorithm}")
    
    # Δημιουργία και εκπαίδευση του ταξινομητή
    clf = LegalDocumentClassifier(algorithm=algorithm)
    _ = clf.train(train_texts, train_labels, max_features=1000)
    
    # Πρόβλεψη για τα δείγματα δοκιμής
    preds = clf.predict(test_texts)
    
    # Υπολογισμός μετρικών
    metrics = metrics_calc.calculate(test_labels, preds, class_names)
    
    print(f"  Ακρίβεια: {metrics['accuracy']:.4f}")
    print(f"  F1 Score: {metrics['f1_score']:.4f}")
    
    # Προσθήκη στα αποτελέσματα
    model_results.append((algorithm, metrics))

In [ ]:
# Σύγκριση των μοντέλων
metrics_calc.compare_models(model_results)

## 6. Αποθήκευση του Μοντέλου και των Μετρικών

Θα αποθηκεύσουμε το καλύτερο μοντέλο και τις μετρικές απόδοσης για μελλοντική χρήση.

In [ ]:
# Επιλογή του καλύτερου αλγορίθμου (υποθέτουμε ότι είναι το SVM)
best_classifier = LegalDocumentClassifier(algorithm='svm')
_ = best_classifier.train(train_texts, train_labels, max_features=1000)

# Δημιουργία του φακέλου για τα μοντέλα αν δεν υπάρχει
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Αποθήκευση του μοντέλου
model_path = models_dir / 'legal_classifier_svm.joblib'
best_classifier.save_model(str(model_path))
print(f"Το μοντέλο αποθηκεύτηκε στο: {model_path}")

# Αποθήκευση των μετρικών
metrics_path = metrics_calc.log_results(metrics_results, "svm")
print(f"Οι μετρικές αποθηκεύτηκαν στο: {metrics_path}")

# Αποθήκευση των μετρικών απόδοσης
performance_path = performance_tracker.save_to_file()
print(f"Οι μετρικές απόδοσης αποθηκεύτηκαν στο: {performance_path}")

## 7. Οπτικοποίηση των Μετρικών Απόδοσης

Θα δημιουργήσουμε γραφήματα για την οπτικοποίηση των μετρικών απόδοσης.

In [ ]:
# Δημιουργία του φακέλου για τις εικόνες αν δεν υπάρχει
images_dir = project_root / 'docs' / 'images'
images_dir.mkdir(parents=True, exist_ok=True)

# Οπτικοποίηση των μετρικών απόδοσης
performance_plot_path = images_dir / 'performance_metrics.png'
performance_tracker.plot_metrics(str(performance_plot_path))

# Οπτικοποίηση του πίνακα σύγχυσης
confusion_matrix_path = images_dir / 'confusion_matrix.png'
metrics_calc.plot_confusion_matrix(metrics_results, class_names, "SVM Classifier", str(confusion_matrix_path))

# Οπτικοποίηση της σύγκρισης των μοντέλων
comparison_path = images_dir / 'model_comparison.png'
metrics_calc.compare_models(model_results, str(comparison_path))

## 8. Συμπεράσματα και Επόμενα Βήματα

Σε αυτό το notebook επιδείξαμε τις βασικές λειτουργίες του συστήματος CDL6000 για την ανάλυση και ταξινόμηση νομικών εγγράφων. Συγκεκριμένα:

1. Πραγματοποιήσαμε προεπεξεργασία νομικών εγγράφων και εξάγαμε βασικά χαρακτηριστικά
2. Εντοπίσαμε και εξάγαμε νομικούς όρους και παραπομπές από τα έγγραφα
3. Εκπαιδεύσαμε και αξιολογήσαμε διαφορετικούς αλγόριθμους ταξινόμησης
4. Παρακολουθήσαμε την απόδοση του συστήματος και μετρήσαμε τη χρήση μνήμης
5. Αποθηκεύσαμε το μοντέλο και τις μετρικές για μελλοντική χρήση
6. Οπτικοποιήσαμε τα αποτελέσματα με γραφήματα

### Επόμενα Βήματα

1. **Βελτίωση του Μοντέλου**: Εκπαίδευση με μεγαλύτερο dataset νομικών εγγράφων
2. **Βελτιστοποίηση Απόδοσης**: Περαιτέρω βελτιστοποίηση για μείωση της χρήσης μνήμης και επιτάχυνση της επεξεργασίας
3. **Επέκταση Χαρακτηριστικών**: Προσθήκη εξαγωγής επιπλέον νομικών χαρακτηριστικών και οντοτήτων
4. **Ενσωμάτωση Deep Learning**: Δοκιμή μοντέλων βαθιάς μάθησης όπως BERT για νομικά κείμενα
5. **Web Interface**: Ανάπτυξη διαδικτυακής διεπαφής για εύκολη χρήση του συστήματος